<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大语言模型》（Build a Large Language Model From Scratch）</a> 一书的配套代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 使用 Ollama 创建偏好数据集

- 偏好微调是一种将指令微调 LLM 与人类偏好对齐的过程
- 为 LLM 偏好微调创建数据集有多种方式
  1. 使用指令微调 LLM 生成多个回复，由人类根据偏好和/或给定偏好标准进行排序
  2. 使用指令微调 LLM 生成多个回复，由 LLM 根据给定偏好标准进行排序
  3. 使用 LLM 根据特定偏好标准生成 preferred 和 dispreferred 回复
- 本 notebook 采用方式 3
- 本 notebook 通过 ollama 使用 700 亿参数的 Llama 3.1-Instruct 模型，为指令数据集生成偏好标签
- 指令数据集的预期格式如下：


### 输入

```json
[
    {
        "instruction": "What is the state capital of California?",
        "input": "",
        "output": "The state capital of California is Sacramento.",
    },
    {
        "instruction": "Provide a synonym for 'fast'.",
        "input": "",
        "output": "A synonym for 'fast' is 'quick'.",
    },
    {
        "instruction": "What is the capital of Greece?",
        "input": "",
        "output": "The capital of Greece is Athens.",

    },
...
]
```

输出数据集如下所示，其中更礼貌的回复为 preferred（`'chosen'`），更不礼貌的回复为 dispreferred（`'rejected'`）：

```json
[
    {
        "instruction": "What is the state capital of California?",
        "input": "",
        "output": "The state capital of California is Sacramento.",
        "rejected": "Look, the state capital of California is obviously Sacramento.",
        "chosen": "The state capital of California is Sacramento."
    },
    {
        "instruction": "Provide a synonym for 'fast'.",
        "input": "",
        "output": "A synonym for 'fast' is 'quick'.",
        "chosen": "A suitable alternative to 'fast' would be 'quick'.",
        "rejected": "A synonym for 'fast' is 'quick'."
    },
    {
        "instruction": "What is the capital of Greece?",
        "input": "",
        "output": "The capital of Greece is Athens.",
        "chosen": "I'd be happy to help! The capital of Greece is indeed Athens.",
        "rejected": "The capital of Greece is Athens."
    },
...
]
```

### 输出




- 代码无需 GPU，在内存足够的笔记本电脑上即可运行

In [ ]:
from importlib.metadata import version

pkgs = ["tqdm",    # 进度条
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

## 安装 Ollama 并下载 Llama 3.1

- Ollama 是一款高效运行 LLM 的应用
- 它是 [llama.cpp](https://github.com/ggerganov/llama.cpp) 的封装，后者用纯 C/C++ 实现 LLM 以最大化效率
- 注意，它是用于 LLM 文本生成（推理）的工具，而非训练或微调 LLM
- 运行下方代码前，请访问 [https://ollama.com](https://ollama.com) 并按说明安装 ollama（例如点击「Download」按钮，下载适用于您操作系统的 ollama 应用）

- macOS 和 Windows 用户：点击您下载的 ollama 应用；若提示安装命令行工具，请选择「yes」
- Linux 用户：可使用 ollama 网站提供的安装命令

- 通常，要从命令行使用 ollama，需要先启动 ollama 应用，或在单独终端中运行 `ollama serve`

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/ollama-eval/ollama-serve.webp?1">


- 在 ollama 应用或 `ollama serve` 运行后，在另一个终端的命令行中执行以下命令试用 700 亿参数的 Llama 3.1 模型

```bash
# 70B model
ollama run llama3.1:70b
```


输出大致如下：

```
$ ollama run llama3.1:70b
pulling manifest
pulling aa81b541aae6... 100% ▕████████████████▏ 39 GB
pulling 8cf247399e57... 100% ▕████████████████▏ 1.7 KB
pulling f1cd752815fc... 100% ▕████████████████▏ 12 KB
pulling 56bb8bd477a5... 100% ▕████████████████▏ 96 B
pulling 3c1c2d3df5b3... 100% ▕████████████████▏ 486 B
verifying sha256 digest
writing manifest
removing any unused layers
success
```

- 注意，`llama3.1:70b` 指指令微调版 700 亿参数的 Llama 3.1 模型

- 或者，您也可以使用更小、更省资源的 80 亿参数 Llama 3.1 模型，将 `llama3.1:70b` 替换为 `llama3.1` 即可

- 下载完成后，您将看到命令行提示符，可与模型对话

- 尝试输入 "What do llamas eat?" 等提示，应返回类似以下输出：

```
>>> What do llamas eat?
Llamas are ruminant animals, which means they have a four-chambered 
stomach and eat plants that are high in fiber. In the wild, llamas 
typically feed on:
1. Grasses: They love to graze on various types of grasses, including tall 
grasses, wheat, oats, and barley.
```

- 可使用 `/bye` 结束会话

## 使用 Ollama 的 REST API

- 现在，与模型交互的另一种方式是通过 Python 调用其 REST API，使用以下函数
- 运行本 notebook 后续单元格前，请确保 ollama 仍在运行，方式同上：
  - 在终端中运行 `ollama serve`
  - 或启动 ollama 应用
- 接下来，运行以下代码单元格查询模型

- 首先用简单示例测试 API，确保其按预期工作：

In [ ]:
import json
import requests


def query_model(prompt, model="llama3.1:70b", url="http://localhost:11434/api/chat"):
    # 将数据 payload 创建为字典
    data = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "options": {
            "seed": 123,
            "temperature": 0,
        }
    }

    # 发送 POST 请求
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data


result = query_model("What do Llamas eat?")
print(result)

## 加载 JSON 条目

- 现在进入数据生成部分
- 此处为动手示例，我们使用第 7 章中原本用于指令微调模型的 `instruction-data.json` 文件：

In [ ]:
from pathlib import Path

json_file = Path("..", "01_main-chapter-code", "instruction-data.json")

with open(json_file, "r") as file:
    json_data = json.load(file)

print("条目数量:", len(json_data))

- 该文件结构如下，其中包含测试数据集中的给定回复（`'output'`），这是我们在指令微调中基于 `'input'` 和 `'instruction'` 训练模型生成的回复：

In [ ]:
json_data[0]

- 下面是一个小型工具函数，用于格式化 instruction 和 input：

In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    instruction_text + input_text

    return instruction_text + input_text

- 现在，让我们尝试使用 ollama API 为偏好微调模型生成 `'chosen'` 和 `'rejected'` 回复
- 此处为演示目的，我们创建更礼貌或更不礼貌的回答


In [ ]:
import random


for entry in json_data[:5]:
    
    politeness = random.choice(["polite", "impolite"])    
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"slightly rewrite the output to be more {politeness}."
        "Keep the modification minimal."
        "Only return return the generated response and nothing else."
    )
    print("\n数据集回复:")
    print(">>", entry['output'])
    print(f"\n{politeness} 回复:")
    print(">>", query_model(prompt))    

- 若上述生成的回复看起来合理，我们可以进入下一步，将该 prompt 应用于整个数据集
- 此处我们为 preferred 回复添加 `'chosen'` 键，为 dispreferred 回复添加 `'rejected'` 键

In [ ]:
import random
from tqdm import tqdm

def generate_model_responses(json_data):

    for i, entry in enumerate(tqdm(json_data, desc="正在写入条目")):
        politeness = random.choice(["polite", "impolite"])    
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"slightly rewrite the output to be more {politeness}."
            "Keep the modification minimal."
            "Only return return the generated response and nothing else."
        )
        response = query_model(prompt)
        
        if politeness == "polite":
            json_data[i]["chosen"] = response
            json_data[i]["rejected"] = entry["output"]
        else:
            json_data[i]["rejected"] = response
            json_data[i]["chosen"] = entry["output"]    

- 现在对整个数据集应用该流程（在 M3 MacBook Air 笔记本电脑上大约需要 17 分钟）
- 注意，ollama 在不同操作系统上并非完全确定性（截至本文撰写时），因此您获得的结果可能与下面显示的略有不同

In [ ]:
generate_model_responses(json_data)

In [ ]:
with open("instruction-data-with-preference.json", "w") as file:
    json.dump(json_data, file, indent=4)